In [ ]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector

from qiskit.quantum_info import SparsePauliOp, Operator
from qiskit.primitives import StatevectorEstimator

In [ ]:
def create_universal_ansatz(num_qubits, num_layers):
    theta = ParameterVector("θ", num_qubits * 3 * num_layers + num_qubits * 2 * (num_layers - 1))
    ansatz = QuantumCircuit(num_qubits)
    
    param_idx = 0
    for layer in range(num_layers):
        # Single-qubit rotations: Ry, Rz on each qubit
        for q in range(num_qubits):
            ansatz.ry(theta[param_idx], q)
            param_idx += 1
            ansatz.rz(theta[param_idx], q)
            param_idx += 1
        
        # Entangling layer (CX ladder)
        if layer < num_layers - 1:
            for q in range(num_qubits - 1):
                ansatz.cx(q, q + 1)
            # Optional: wrap around
            if num_qubits > 2:
                ansatz.cx(num_qubits - 1, 0)
    
    # Final single-qubit rotations
    for q in range(num_qubits):
        ansatz.ry(theta[param_idx], q)
        param_idx += 1
        ansatz.rz(theta[param_idx], q)
        param_idx += 1
    
    return ansatz, theta

# What follows is a usage example
ansatz, theta = create_universal_ansatz(2, num_layers=2)
print(ansatz.draw())

In [ ]:
def hardware_efficient_ansatz(num_qubits, num_layers):
    theta = ParameterVector("θ", num_qubits * 2 * num_layers)
    ansatz = QuantumCircuit(num_qubits)
    
    param_idx = 0
    for layer in range(num_layers):
        for q in range(num_qubits):
            ansatz.ry(theta[param_idx], q)
            param_idx += 1
            ansatz.rz(theta[param_idx], q)
            param_idx += 1
        for q in range(num_qubits - 1):
            ansatz.cx(q, q + 1)
    
    return ansatz, theta

# Usage example
ansatz, theta = hardware_efficient_ansatz(2, num_layers=2)
print(ansatz.draw())

# First just the simple example, no obfuscation 

## Define the Hamiltonian

In [ ]:
# Simple H = X x Y
H = SparsePauliOp("XY")   

## Print tyhe exact spectrum

In [ ]:
# Print exact spectrum
eigvals = np.linalg.eigvalsh(Operator(H).data)
print(f"Exact eigenvalues: {np.round(eigvals, 6)}")
print(f"Exact ground state energy: {eigvals[0]:.6f}\n")

# Hardware efficient ansatz for 2 qubits

In [ ]:
ansatz, theta = hardware_efficient_ansatz(2, num_layers=2)
print(ansatz.draw())

In [ ]:
# Cost function

In [ ]:
estimator = StatevectorEstimator()
def cost_func(params):
    job = estimator.run([(ansatz, H, [params])])
    return float(job.result()[0].data.evs[0])

In [ ]:
# Optimize
np.random.seed(42)
best_result = None

In [ ]:
for trial in range(8):
    x0 = np.random.uniform(0, 2 * np.pi, size=len(theta))
    res = minimize(cost_func, x0, method="cobyla", options={"maxiter": 2000})
    if best_result is None or res.fun < best_result.fun:
        best_result = res

In [ ]:
print(f"Optimized ground state energy : {best_result.fun:.6f}")
print(f"Optimal parameters θ          : {np.round(best_result.x, 4)}")
print(f"Optimization success          : {best_result.success}")